# Approximation Week: Taylor Series, Numerical Integration & RK4

**Goal for today:**
1. Understand why we *approximate* instead of solving exactly
2. Build functions from scratch with **Taylor Series**
3. Approximate area under a curve with **numerical integration** (Riemann sums, Trapezoidal rule)
4. Approximate solutions to differential equations with **Euler's Method** and **RK4**


## 1. Why Approximate?

Last time we split models into **deterministic** and **stochastic**. Today we go deep on the deterministic side.

Most equations you meet in a math class are chosen *because* they have a clean, exact, closed-form answer. Most equations that show up in the real world don't:

- Most functions don't have a simple polynomial formula (`sin`, `e^x`, `ln(x)`...) — computers still need to evaluate them
- Most integrals have no elementary antiderivative (e.g. $\int e^{-x^2}\,dx$)
- Most differential equations can't be solved by hand at all (e.g. orbital mechanics, chemical reactions, circuits)

**The approximation mindset:** trade a small, controllable amount of error for an answer you can actually compute — and make the error *shrink* as you're willing to do more work (more terms, more steps, smaller step size).

That trade-off — accuracy vs. computation — is the theme of every method below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

## 2. Taylor Series — Building Functions from Derivatives

**Core idea:** at a single point $a$, a function's value and *all* its derivatives contain enough information to rebuild the function nearby, one polynomial term at a time.

$$f(x) \approx f(a) + f'(a)(x-a) + \frac{f''(a)}{2!}(x-a)^2 + \frac{f'''(a)}{3!}(x-a)^3 + \cdots$$

Each new term corrects the approximation to match one more derivative at $a$. More terms = a better match, but only for one specific choice of $a$ — and it fades the further $x$ wanders from $a$.

For $\sin(x)$ centered at $a=0$, the odd derivatives alternate $\cos(0), -\sin(0), -\cos(0), \sin(0), \dots$, which collapses to:

$$\sin(x) \approx x - \frac{x^3}{3!} + \frac{x^5}{5!} - \frac{x^7}{7!} + \cdots$$

### Activity 1 — Approximating sin(x) with Taylor Polynomials

We'll build `sin(x)` from scratch (no `np.sin` allowed in this function!) using a growing number of terms, and watch the approximation get better.

In [ ]:
def taylor_sin(x, n_terms):
    """Approximate sin(x) using n_terms of its Taylor series around 0."""
    x = np.asarray(x, dtype=float)
    total = np.zeros_like(x)
    for k in range(n_terms):
        power = 2 * k + 1
        term = ((-1) ** k) * (x ** power) / math.factorial(power)
        total += term
    return total

x = np.linspace(-2 * np.pi, 2 * np.pi, 400)
true_sin = np.sin(x)

plt.figure(figsize=(8, 5))
plt.plot(x, true_sin, color="black", linewidth=2.5, label="true sin(x)")
for n_terms in [1, 2, 3, 5, 10]:
    plt.plot(x, taylor_sin(x, n_terms), linestyle="--", label=f"{n_terms} term(s)")
plt.ylim(-2, 2)
plt.axvline(0, color="gray", linewidth=0.8)
plt.title("Taylor polynomial approximations of sin(x), centered at 0")
plt.xlabel("x")
plt.legend(loc="upper right", fontsize=8)
plt.show()

**Discussion:** notice the approximation hugs the true curve tightly near $x=0$ (the center) even with just 1-2 terms, but needs many more terms to stay accurate as $|x|$ grows. This is the fundamental trade-off of Taylor series: accuracy is best *near the center* and costs more terms the farther out you go.

In [ ]:
# How many terms does it take to nail sin(x) at a point far from the center?
x_far = 2 * np.pi  # two full periods away from a=0
term_counts = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15]
errors = [abs(taylor_sin(x_far, n) - np.sin(x_far)) for n in term_counts]

plt.figure(figsize=(7, 5))
plt.semilogy(term_counts, errors, marker="o")
plt.xlabel("Number of Taylor terms")
plt.ylabel("Absolute error at x = 2π (log scale)")
plt.title("Taylor series error shrinks fast once it 'catches up'")
plt.grid(True, alpha=0.3)
plt.show()

## 3. Numerical Integration — Approximating Area Under a Curve

An integral $\int_a^b f(x)\,dx$ is just the area under a curve. When there's no clean antiderivative, approximate the area directly by slicing it into simple shapes and summing them.

**Riemann sum (rectangles):** slice $[a,b]$ into $n$ strips of width $h=\frac{b-a}{n}$, and approximate each strip's area with a rectangle whose height is $f$ evaluated at the strip's midpoint:

$$\int_a^b f(x)\,dx \approx h\sum_{i=1}^{n} f(x_i^{\text{mid}})$$

**Trapezoidal rule:** same slices, but cap each strip with a straight line instead of a flat top — a trapezoid instead of a rectangle — which usually fits the curve better:

$$\int_a^b f(x)\,dx \approx h\left(\frac{f(a)+f(b)}{2} + \sum_{i=1}^{n-1} f(x_i)\right)$$

Both converge to the exact area as $n\to\infty$ and $h\to 0$.

### Activity 2 — Integrating sin(x) from 0 to π

**Exact answer:** $\int_0^\pi \sin(x)\,dx = [-\cos(x)]_0^\pi = 2$. We know the answer, so we can measure exactly how wrong each approximation is.

In [ ]:
def riemann_midpoint(f, a, b, n):
    h = (b - a) / n
    midpoints = a + (np.arange(n) + 0.5) * h
    return h * np.sum(f(midpoints))

def trapezoidal(f, a, b, n):
    xs = np.linspace(a, b, n + 1)
    ys = f(xs)
    h = (b - a) / n
    return h * (0.5 * ys[0] + 0.5 * ys[-1] + np.sum(ys[1:-1]))

exact = 2.0
n_demo = 8
xs_fill = np.linspace(0, np.pi, 400)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Riemann midpoint rectangles
h = np.pi / n_demo
mids = (np.arange(n_demo) + 0.5) * h
axes[0].plot(xs_fill, np.sin(xs_fill), color="black")
axes[0].bar(mids, np.sin(mids), width=h, alpha=0.4, edgecolor="crimson", color="crimson")
axes[0].set_title(f"Riemann midpoint rectangles (n={n_demo})")

# Trapezoids
xs_t = np.linspace(0, np.pi, n_demo + 1)
axes[1].plot(xs_fill, np.sin(xs_fill), color="black")
axes[1].fill_between(xs_t, np.sin(xs_t), alpha=0.4, color="steelblue", step=None)
axes[1].plot(xs_t, np.sin(xs_t), color="steelblue", marker="o")
axes[1].set_title(f"Trapezoidal rule (n={n_demo})")

plt.tight_layout()
plt.show()

print(f"Exact area: {exact}")
print(f"Riemann midpoint (n={n_demo}): {riemann_midpoint(np.sin, 0, np.pi, n_demo):.5f}")
print(f"Trapezoidal (n={n_demo}):      {trapezoidal(np.sin, 0, np.pi, n_demo):.5f}")

In [ ]:
n_values = [4, 8, 16, 32, 64, 128, 256]
riemann_errors = [abs(riemann_midpoint(np.sin, 0, np.pi, n) - exact) for n in n_values]
trapezoid_errors = [abs(trapezoidal(np.sin, 0, np.pi, n) - exact) for n in n_values]

plt.figure(figsize=(7, 5))
plt.loglog(n_values, riemann_errors, marker="o", label="Riemann midpoint")
plt.loglog(n_values, trapezoid_errors, marker="s", label="Trapezoidal")
plt.xlabel("Number of slices (n)")
plt.ylabel("Absolute error (log scale)")
plt.title("Both rules converge as n grows — much faster than Monte Carlo's 1/√n")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

**Discussion:** compare this to the Monte Carlo error plot from last lesson. Monte Carlo error shrinks like $1/\sqrt{n}$ — a shallow slope. Riemann and Trapezoidal error here shrinks much faster (close to $1/n^2$ — a steep slope), because they exploit the fact that the function is *smooth*, not random. Structure you can exploit is always worth more than brute-force averaging.

## 4. Numerical ODEs — Euler's Method and RK4

A differential equation describes a *rate of change*, not a value directly: $\frac{dy}{dt} = f(t, y)$. Many can't be solved by hand, so instead we walk forward from a known starting point in small time steps.

**Euler's Method** — the simplest possible idea: at each step, follow the current slope in a straight line for one step of size $h$:

$$y_{n+1} = y_n + h \cdot f(t_n, y_n)$$

Cheap, but the slope changes *during* the step and Euler ignores that — error accumulates fast.

**RK4 (4th-order Runge-Kutta)** — same idea, but samples the slope at four cleverly chosen points within the step and combines them as a weighted average, so it "sees" how the slope curves during the step:

$$k_1 = f(t_n, y_n)$$
$$k_2 = f\left(t_n+\tfrac{h}{2},\ y_n+\tfrac{h}{2}k_1\right)$$
$$k_3 = f\left(t_n+\tfrac{h}{2},\ y_n+\tfrac{h}{2}k_2\right)$$
$$k_4 = f(t_n+h,\ y_n+h\,k_3)$$
$$y_{n+1} = y_n + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

Same number of *steps*, dramatically less error — the classic example of why RK4 (not Euler) is the workhorse method.

### Activity 3 — Solving dy/dt = y

**Setup:** $\frac{dy}{dt} = y$, $y(0) = 1$. This has a clean exact solution, $y(t) = e^t$, so we can grade both numerical methods against ground truth.

We'll deliberately use a coarse step size ($h=0.5$) so the difference between methods is dramatic.

In [ ]:
def euler_method(f, t0, y0, h, n_steps):
    ts, ys = [t0], [y0]
    t, y = t0, y0
    for _ in range(n_steps):
        y = y + h * f(t, y)
        t = t + h
        ts.append(t)
        ys.append(y)
    return np.array(ts), np.array(ys)

def rk4_method(f, t0, y0, h, n_steps):
    ts, ys = [t0], [y0]
    t, y = t0, y0
    for _ in range(n_steps):
        k1 = f(t, y)
        k2 = f(t + h / 2, y + h / 2 * k1)
        k3 = f(t + h / 2, y + h / 2 * k2)
        k4 = f(t + h, y + h * k3)
        y = y + (h / 6) * (k1 + 2 * k2 + 2 * k3 + k4)
        t = t + h
        ts.append(t)
        ys.append(y)
    return np.array(ts), np.array(ys)

f = lambda t, y: y  # dy/dt = y
t0, y0, h, n_steps = 0, 1, 0.5, 10

ts_euler, ys_euler = euler_method(f, t0, y0, h, n_steps)
ts_rk4, ys_rk4 = rk4_method(f, t0, y0, h, n_steps)
ts_exact = np.linspace(0, 5, 200)
ys_exact = np.exp(ts_exact)

plt.figure(figsize=(8, 5))
plt.plot(ts_exact, ys_exact, color="black", linewidth=2, label="exact: y = e^t")
plt.plot(ts_euler, ys_euler, marker="o", linestyle="--", color="crimson", label="Euler's Method")
plt.plot(ts_rk4, ys_rk4, marker="s", linestyle="--", color="seagreen", label="RK4")
plt.xlabel("t")
plt.ylabel("y")
plt.title(f"dy/dt = y, y(0)=1, step size h={h}")
plt.legend()
plt.show()

print(f"Exact y(5):  {np.exp(5):.4f}")
print(f"Euler y(5):  {ys_euler[-1]:.4f}  (error: {abs(ys_euler[-1]-np.exp(5)):.4f})")
print(f"RK4 y(5):    {ys_rk4[-1]:.4f}  (error: {abs(ys_rk4[-1]-np.exp(5)):.4f})")

In [ ]:
step_sizes = [0.5, 0.25, 0.1, 0.05, 0.025, 0.01]
euler_errors, rk4_errors = [], []

for h_test in step_sizes:
    n_steps_test = int(5 / h_test)
    _, ys_e = euler_method(f, 0, 1, h_test, n_steps_test)
    _, ys_r = rk4_method(f, 0, 1, h_test, n_steps_test)
    euler_errors.append(abs(ys_e[-1] - np.exp(5)))
    rk4_errors.append(abs(ys_r[-1] - np.exp(5)))

plt.figure(figsize=(7, 5))
plt.loglog(step_sizes, euler_errors, marker="o", label="Euler's Method")
plt.loglog(step_sizes, rk4_errors, marker="s", label="RK4")
plt.xlabel("Step size h (log scale)")
plt.ylabel("Absolute error at t=5 (log scale)")
plt.title("Error vs. step size: Euler (order 1) vs. RK4 (order 4)")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.gca().invert_xaxis()
plt.show()

**Discussion:** look at the slopes on that log-log plot. Euler's error drops proportionally to $h$ (order 1) — halve the step size, halve the error. RK4's error drops proportionally to $h^4$ (order 4) — halve the step size, and the error shrinks by a factor of $2^4=16$. That's why RK4 can use large, cheap steps and still beat Euler using tiny, expensive ones.

## 5. Key Takeaways

- **Approximation trades exactness for computability** — use more terms/steps/slices to buy back accuracy.
- **Taylor series** rebuilds a function from its derivatives at one point; accuracy is best near the center and needs more terms farther away.
- **Numerical integration** (Riemann, Trapezoidal) approximates area by slicing it into simple shapes; error shrinks fast ($\sim 1/n^2$) because it exploits smoothness — much faster than Monte Carlo's $1/\sqrt{n}$.
- **Euler's Method** takes straight-line steps along the current slope — simple but error-prone (order 1).
- **RK4** samples the slope 4 times per step and averages them — same cost per step, vastly better accuracy (order 4).
- The recurring pattern across all of these: **more work → predictably less error**, and the *rate* at which error shrinks is what separates a good method from a great one.

## 6. Try It Yourself (Exercises)

1. **Modify Activity 1** to build `taylor_cos(x, n_terms)` from scratch using the Taylor series for $\cos(x)$. Plot it against `np.cos` the same way.
2. **Extend Activity 2** to integrate $f(x) = e^{-x^2}$ from $-2$ to $2$ (this has no elementary antiderivative — numerical integration is the *only* way). Compare Riemann vs. Trapezoidal at a few values of $n$.
3. **New problem:** Use Euler's Method and RK4 to solve $\frac{dy}{dt} = -y$ (exponential decay), $y(0)=10$, and compare both to the exact solution $y(t)=10e^{-t}$.
4. **Challenge:** In Activity 3, how small does Euler's step size $h$ need to be before its error at $t=5$ finally beats RK4's error at $h=0.5$? Use the error-vs-h plot to estimate it, then verify with code.